# Week 02 · HTTP、REST 与 FastAPI

浏览器是客户端，API 是服务器。HTTP 请求由方法、URL、头和可选请求体组成；响应由状态码、头和内容组成。GET 应只读取资源，POST 创建资源，PATCH 局部更新，DELETE 删除。201 表示已创建，204 表示成功且无响应体，404 表示目标不存在，422 表示输入不满足结构约束。错误不能总返回 200，否则调用者无法可靠判断失败。

FastAPI 路由将 HTTP 输入映射到函数。Pydantic 负责形状、类型和范围校验，不负责所有业务规则。`response_model` 限制响应字段，避免无意泄露内部数据。测试客户端在进程内执行真实路由与校验，不需要占用端口。内存字典只适合这周学习，进程重启就丢失；下一周迁移数据库。

pytest 的测试必须独立，fixture 可以为每个测试创建空存储；成功路径、边界和错误路径同样重要。不要用测试依赖执行顺序掩盖状态问题。运行 `uvicorn module:app` 后可以在 `/docs` 试 API；Notebook 中运行的是 TestClient，不会偷偷开启公网服务。

## 学习方式 / How to study
先预测代码结果，再逐行运行。改变一个输入、解释变化，最后不看参考实现重写关键函数。阅读不是掌握的证据；能独立实现、测试、解释失败才是。

In [ ]:
from fastapi import FastAPI, HTTPException, Response
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class TaskCreate(BaseModel):
    title: str = Field(min_length=1, max_length=120)

class TaskPatch(BaseModel):
    done: bool

def make_api():
    app = FastAPI(title="Learning Task API")
    storage = {}

    @app.post("/tasks", status_code=201)
    def create_task(body: TaskCreate):
        if not body.title.strip():
            raise HTTPException(422, "Title cannot be blank")
        identifier = max(storage, default=0)+1
        storage[identifier] = {"id": identifier, "title": body.title, "done": False}
        return storage[identifier]

    @app.get("/tasks")
    def list_tasks(done: bool | None = None, limit: int = 20):
        if not 1 <= limit <= 100:
            raise HTTPException(422, "limit must be 1..100")
        return [t for t in storage.values() if done is None or t["done"] == done][:limit]

    @app.get("/tasks/{identifier}")
    def get_task(identifier: int):
        if identifier not in storage:
            raise HTTPException(404, "Task not found")
        return storage[identifier]

    @app.patch("/tasks/{identifier}")
    def patch_task(identifier: int, body: TaskPatch):
        task = get_task(identifier)
        task["done"] = body.done
        return task

    @app.delete("/tasks/{identifier}", status_code=204)
    def delete_task(identifier: int):
        get_task(identifier)
        del storage[identifier]
        return Response(status_code=204)

    return app

with TestClient(make_api()) as client:
    result = client.post("/tasks", json={"title": "Read HTTP docs"})
    assert result.status_code == 201
    identifier = result.json()["id"]
    assert client.get(f"/tasks/{identifier}").json()["done"] is False
    assert client.post("/tasks", json={"title": ""}).status_code == 422
    assert client.get("/tasks/999").status_code == 404
    assert client.patch(f"/tasks/{identifier}", json={"done": True}).json()["done"]
    assert len(client.get("/tasks?done=true").json()) == 1
    assert client.delete(f"/tasks/{identifier}").status_code == 204
    assert client.get(f"/tasks/{identifier}").status_code == 404
    print("8 个真实 HTTP 断言通过；OpenAPI 路径：", list(client.get("/openapi.json").json()["paths"]))

## 练习 / Exercises
增加分页 offset，非法负数返回 422。练习为每项行为写独立 pytest 函数。解释 PATCH 和 PUT 的差别。

先在下面独立完成，再展开参考实现。

In [ ]:
# 在这里写你的实现；运行后检查边界。


## 参考实现与验收 / Reference and checks
参考实现是一个可行方案，不是唯一答案。不要在未完成练习前直接复制。

In [ ]:
# 测试之间新建应用，确保存储隔离。
with TestClient(make_api()) as fresh_client:
    assert fresh_client.get("/tasks").json() == []
    assert fresh_client.get("/tasks?limit=0").status_code == 422
    assert fresh_client.post("/tasks", json={"title": " "}).status_code == 422
print("隔离、边界与空白输入测试通过")